# Comparison: `initTurbulence` vs `SimpleGridTurbulence`

Side-by-side comparison of legacy and modern CRPropa3 turbulent field generation APIs.

In [ ]:
# === Imports and Configuration ===

import healpy as hp
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import pylab 
import matplotlib as mpl
from math import *
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import cm
import matplotlib as mpl
mpl.rc('image', cmap='jet')
from crpropa import *
from scipy.fft import fft,fftn,fftfreq
import matplotlib
import os
import sys
import joblib as jb
matplotlib.rcParams.update({'font.size': 25})
from collections import OrderedDict
import palettable as pt
from datetime import date
today = date.today()
import time
today = today.strftime("%b-%d-%Y")
print("today =", today)
from matplotlib.pyplot import figure
deg     =  np.degrees
rad     =  np.radians
log10   =  np.log10
fnan    = float('nan')

## Generate Both Fields

In [ ]:
%%time
# === Generate both turbulent B-fields for comparison ===

bTur = 6
randomSeed = 10
lmin = 450*pc
lmax = 4000*pc

# --- Legacy API: initTurbulence ---
vgrid = Grid3f(Vector3d(10*kpc,10*kpc,10*kpc), 201, 100*pc)
initTurbulence(vgrid, bTur*muG, lmin, lmax, -11./3., randomSeed)
bField_old = MagneticFieldGrid(vgrid)
Lc  = turbulentCorrelationLength(lmin/pc, lmax/pc)
print ('--- Legacy initTurbulence ---')
print ('Lc = %.1f pc' % turbulentCorrelationLength(lmin/pc, lmax/pc, -11./3.) ) # correlation length
print ('sqrt(<B^2>) = %.1f muG' % (rmsFieldStrength(vgrid) / muG) )  # RMS
print ('<|B|> = %.1f muG' % (meanFieldStrength(vgrid) / muG) ) # mean
print ('B(10 kpc, 0, 0) =', bField_old.getField(Vector3d(10,0,0) * kpc) / muG, 'muG')
print ('B(20 kpc, 0, 0) =', bField_old.getField(Vector3d(20,0,0) * kpc) / muG, 'muG')
print ('B(0 kpc, 0, 0) =', bField_old.getField(Vector3d(0,0,0) * kpc) / muG, 'muG')

# --- Modern API: SimpleGridTurbulence ---
# Requires SimpleTurbulenceSpectrum + GridProperties (not raw Grid3f)
spectrum = SimpleTurbulenceSpectrum(bTur * muG, lmin, lmax, -11./3.)
gridProp = GridProperties(Vector3d(10*kpc, 10*kpc, 10*kpc), 201, 100*pc)
bField_tur = SimpleGridTurbulence(spectrum, gridProp, randomSeed)
Lc = bField_tur.getCorrelationLength()
print ('\n--- Modern SimpleGridTurbulence ---')
print ('Lc = %.1f pc' % turbulentCorrelationLength(lmin/pc, lmax/pc, -11./3.) ) # correlation length
print ('sqrt(<B^2>) = %.1f muG' % (rmsFieldStrength(vgrid) / muG) )  # RMS
print ('<|B|> = %.1f muG' % (meanFieldStrength(vgrid) / muG) ) # mean
print ('B(10 kpc, 0, 0) =', bField_tur.getField(Vector3d(10,0,0) * kpc) / muG, 'muG')
print ('B(20 kpc, 0, 0) =', bField_tur.getField(Vector3d(20,0,0) * kpc) / muG, 'muG')
print ('B(0 kpc, 0, 0) =', bField_tur.getField(Vector3d(0,0,0) * kpc) / muG, 'muG')

## |B| Histogram Comparison

In [ ]:
# === Compare |B| distributions along the x-axis for both APIs ===

import numpy as np
positions = [Vector3d(x*kpc, 0, 0) for x in np.linspace(-10, 10, 100)]
B_old = [bField_old.getField(p).getR() / muG for p in positions]
B_new = [bField_tur.getField(p).getR() / muG for p in positions]
plt.figure(figsize=(10, 10))
plt.hist(B_old, bins=30, alpha=0.5, label='initTurbulence')
plt.hist(B_new, bins=30, alpha=0.5, label='SimpleGridTurbulence')
plt.xlabel('|B| [muG]')
plt.legend()
plt.show()

## Sample Both Fields on 3D Grid

In [ ]:
%%time
# === Sample both B-fields on a uniform 3D grid ===

dr = 10*pc
len_x,len_y,len_z = 200,200,200
x = np.linspace(1,1000,len_x)*dr
y = np.linspace(1,1000,len_y)*dr
z = np.linspace(1,1000,len_z)*dr
# x = np.logspace(0,3.,len_x)*dr
# y = np.logspace(0,3.,len_y)*dr
# z = np.logspace(0,3.,len_y)*dr

# --- Legacy field ---
B_tur_old = np.zeros([len_x,len_y,len_z,3])
for i in range(0,len_x):
    for j in range(0,len_y):
        for k in range(0,len_z):
            pos = Vector3d(x[i],y[j],z[k])
            B =  (bField_old.getField(pos) / gauss) # B in [G]
            B_tur_old[i,j,k,:] = np.array(B)

# --- Modern field ---
B_tur_new = np.zeros([len_x,len_y,len_z,3])
for i in range(0,len_x):
    for j in range(0,len_y):
        for k in range(0,len_z):
            pos = Vector3d(x[i],y[j],z[k])
            B =  (bField_tur.getField(pos) / gauss) # B in [G]
            B_tur_new[i,j,k,:] = np.array(B)

## 1D Directional Power Spectrum Function

In [ ]:
# === 1D Directional Power Spectra via Manual DFT ===
# Compute power along each axis by averaging over multiple parallel lines,
# offsetting indices to avoid correlations between sampled lines.

import numpy as np
from numpy import pi, cos, sin
def compute_1D_power_spectra(B_tur, x, y, z, lambdas, ind=80):
    dx = x[1] - x[0]
    dy = y[1] - y[0]
    dz = z[1] - z[0]
    
    len_lambda = len(lambdas)
    PS_x = np.zeros((len_lambda, 3))
    PS_y = np.zeros((len_lambda, 3))
    PS_z = np.zeros((len_lambda, 3))
    
    nx, ny, nz, _ = B_tur.shape
    for i, lambda_i in enumerate(lambdas):
        # --- X-direction ---
        arg_x = 2 * pi * x / lambda_i
        cos_x = cos(arg_x)
        sin_x = sin(arg_x)
        powers_x = []
        for o in range(ind):
            if o+2 >= ny or o >= nz:
                continue
            B_line = B_tur[:, o+2, o, :]  # shape (nx, 3)
            real_part = np.sum(B_line * cos_x[:, None] * dx, axis=0)
            imag_part = np.sum(B_line * sin_x[:, None] * dx, axis=0)
            fft_val = real_part - 1j * imag_part
            powers_x.append(np.abs(fft_val)**2)
            # print(B_line.shape,np.array(powers_x).shape, fft_val.shape)
        if powers_x:
            PS_x[i, :] = np.mean(powers_x, axis=0)
        # --- Y-direction ---
        arg_y = 2 * pi * y / lambda_i
        cos_y = cos(arg_y)
        sin_y = sin(arg_y)
        powers_y = []
        for o in range(ind):
            if o+5 >= nz or o >= nx:
                continue
            B_line = B_tur[o, :, o+5, :]
            real_part = np.sum(B_line * cos_y[:, None] * dy, axis=0)
            imag_part = np.sum(B_line * sin_y[:, None] * dy, axis=0)
            fft_val = real_part - 1j * imag_part
            powers_y.append(np.abs(fft_val)**2)
        if powers_y:
            PS_y[i, :] = np.mean(powers_y, axis=0)
        # --- Z-direction ---
        arg_z = 2 * pi * z / lambda_i
        cos_z = cos(arg_z)
        sin_z = sin(arg_z)
        powers_z = []
        for o in range(ind):
            if o+5 >= ny or o >= nx:
                continue
            B_line = B_tur[o, o+5, :, :]
            real_part = np.sum(B_line * cos_z[:, None] * dz, axis=0)
            imag_part = np.sum(B_line * sin_z[:, None] * dz, axis=0)
            fft_val = real_part - 1j * imag_part
            powers_z.append(np.abs(fft_val)**2)
        if powers_z:
            PS_z[i, :] = np.mean(powers_z, axis=0)
    return PS_x, PS_y, PS_z

## 1D Power Spectra Overlay: wavelength in parsecs

In [ ]:
# === 1D Power Spectra Comparison: wavelength in parsecs ===

bins_per_decade = 300
len_lambda = int(bins_per_decade * (log10(lmax) - log10(lmin)))
lambdas = np.logspace(log10(lmin),log10(lmax),len_lambda)

# Legacy
PS_x_old, PS_y_old, PS_z_old = compute_1D_power_spectra(B_tur_old, x, y, z, lambdas, ind=80)
PS_x_total_old = np.sum(PS_x_old, axis=1)
PS_y_total_old = np.sum(PS_y_old, axis=1)
PS_z_total_old = np.sum(PS_z_old, axis=1)

# Modern
PS_x_new, PS_y_new, PS_z_new = compute_1D_power_spectra(B_tur_new, x, y, z, lambdas, ind=80)
PS_x_total_new = np.sum(PS_x_new, axis=1)
PS_y_total_new = np.sum(PS_y_new, axis=1)
PS_z_total_new = np.sum(PS_z_new, axis=1)

# Kolmogorov reference slope
L_ref = lambdas[len(lambdas)//3]
P_ref = PS_x_total_old[len(lambdas)//3]
slope_line = P_ref * (lambdas / L_ref)**(5/3)

k_vals = 1 / lambdas

plt.figure(figsize=(12, 10))
# Legacy
plt.loglog(lambdas/pc, PS_x_total_old, label='initTurbulence x', alpha=0.7)
plt.loglog(lambdas/pc, PS_y_total_old, label='initTurbulence y', alpha=0.7)
plt.loglog(lambdas/pc, PS_z_total_old, label='initTurbulence z', alpha=0.7)
# Modern
plt.loglog(lambdas/pc, PS_x_total_new, '--', label='SimpleGridTurbulence x', alpha=0.7)
plt.loglog(lambdas/pc, PS_y_total_new, '--', label='SimpleGridTurbulence y', alpha=0.7)
plt.loglog(lambdas/pc, PS_z_total_new, '--', label='SimpleGridTurbulence z', alpha=0.7)
# Reference
plt.loglog(lambdas/pc, slope_line, '--k', label=r'$L^{5/3}$ reference')
# plt.xlabel(r'$k$ (1/pc)')
plt.xlabel(r'$L$ [pc]')
plt.ylabel(r'$P(k)$')
plt.title('1D Power Spectra Comparison: initTurbulence vs SimpleGridTurbulence')
plt.legend()
plt.grid(True, which='both', ls='--')
plt.tight_layout()
plt.show()

## 1D Power Spectra Overlay: wavelength in CRPropa units

In [ ]:
# === 1D Power Spectra Comparison: wavelength in CRPropa internal units ===

bins_per_decade = 300
len_lambda = int(bins_per_decade * (log10(lmax) - log10(lmin)))
lambdas = np.logspace(log10(lmin),log10(lmax),len_lambda)

# Legacy
PS_x_old, PS_y_old, PS_z_old = compute_1D_power_spectra(B_tur_old, x, y, z, lambdas, ind=80)
PS_x_total_old = np.sum(PS_x_old, axis=1)
PS_y_total_old = np.sum(PS_y_old, axis=1)
PS_z_total_old = np.sum(PS_z_old, axis=1)

# Modern
PS_x_new, PS_y_new, PS_z_new = compute_1D_power_spectra(B_tur_new, x, y, z, lambdas, ind=80)
PS_x_total_new = np.sum(PS_x_new, axis=1)
PS_y_total_new = np.sum(PS_y_new, axis=1)
PS_z_total_new = np.sum(PS_z_new, axis=1)

# Kolmogorov reference slope
L_ref = lambdas[len(lambdas)//3]
P_ref = PS_x_total_old[len(lambdas)//3]
slope_line = P_ref * (lambdas / L_ref)**(5/3)

plt.figure(figsize=(12, 10))
# Legacy
plt.loglog(lambdas, PS_x_total_old, label='initTurbulence x', alpha=0.7)
plt.loglog(lambdas, PS_y_total_old, label='initTurbulence y', alpha=0.7)
plt.loglog(lambdas, PS_z_total_old, label='initTurbulence z', alpha=0.7)
# Modern
plt.loglog(lambdas, PS_x_total_new, '--', label='SimpleGridTurbulence x', alpha=0.7)
plt.loglog(lambdas, PS_y_total_new, '--', label='SimpleGridTurbulence y', alpha=0.7)
plt.loglog(lambdas, PS_z_total_new, '--', label='SimpleGridTurbulence z', alpha=0.7)
# Reference
plt.loglog(lambdas, slope_line, '--k', label=r'$L^{5/3}$ reference')
plt.xlabel(r'$L$ (pc)')
plt.ylabel(r'$P(k)$')
plt.title('1D Power Spectra Comparison: initTurbulence vs SimpleGridTurbulence')
plt.legend()
plt.grid(True, which='both', ls='--')
plt.tight_layout()
plt.show()

## 3D Isotropic Power Spectrum Comparison

In [ ]:
# === 3D Isotropic Power Spectrum Comparison ===

import numpy as np
from numpy.fft import fftn, fftfreq
from scipy.stats import binned_statistic

np.random.seed(0)
nx, ny, nz = 200, 200, 200
Lx, Ly, Lz = 10.0, 10.0, 10.0
dx, dy, dz = Lx/nx, Ly/ny, Lz/nz

def compute_3d_iso_ps(B_field):
    fft_B = np.zeros((nx, ny, nz, 3), dtype=complex)
    for i in range(3):
        fft_B[..., i] = fftn(B_field[..., i], norm='forward')
    power_spectrum_3d = np.sum(np.abs(fft_B)**2, axis=3)
    kx = fftfreq(nx, d=dx) * 2 * np.pi
    ky = fftfreq(ny, d=dy) * 2 * np.pi
    kz = fftfreq(nz, d=dz) * 2 * np.pi
    KX, KY, KZ = np.meshgrid(kx, ky, kz, indexing='ij')
    k_mag = np.sqrt(KX**2 + KY**2 + KZ**2).flatten()
    power_flat = power_spectrum_3d.flatten()
    k_bins = np.logspace(np.log10(np.min(k_mag[k_mag > 0])), np.log10(np.max(k_mag)), 50)
    bin_means, bin_edges, _ = binned_statistic(k_mag, power_flat, bins=k_bins, statistic='mean')
    k_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    return k_centers, bin_means

k_old, ps_old = compute_3d_iso_ps(B_tur_old)
k_new, ps_new = compute_3d_iso_ps(B_tur_new)

plt.figure(figsize=(10, 8))
plt.loglog(k_old, ps_old, label='initTurbulence')
plt.loglog(k_new, ps_new, '--', label='SimpleGridTurbulence')
plt.xlabel(r'$k$')
plt.ylabel(r'$P(k)$')
plt.title('3D Isotropic Power Spectrum Comparison')
plt.grid(True, which='both', ls='--')
plt.legend()
plt.tight_layout()
plt.show()